In [0]:
%pip install librosa

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 26.6 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
storage_account_name = ""
storage_account_key = ""

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)


In [0]:
metadata_path = f"abfss://processed@{storage_account_name}.dfs.core.windows.net/911-recordings/metadata_raw"

metadata_df = spark.read.parquet(metadata_path)

print(f"Rows    : {metadata_df.count()}")
print(f"Columns : {metadata_df.columns}")
display(metadata_df.limit(5))

Rows    : 704
Columns : ['id', 'link', 'title', 'date', 'state', 'civilian_initiated', 'deaths', 'potential_death', 'false_alarm', 'description', 'file_name']


id,link,title,date,state,civilian_initiated,deaths,potential_death,false_alarm,description,file_name
1,https://web.archive.org/web/20150417085342/http://mp3.911dispatch.com.s3.amazonaws.com/northhollywood_radio.mp3,North Hollywood bank robbery,2/97,California,0,2,1,0,"– The unforgettable collection of radio logging tapes from the 1997 violent robbery of the Bank of America in Los Angeles. The radio traffic begins routinely, then an officer passing the bank notices the robbers and radios in “shots fired.” Then all breaks loose.",call_1.mp3
2,https://web.archive.org/web/20150417085342/http://mp3.911dispatch.com.s3.amazonaws.com/detroit_911_1.mp3,Detroit Child’s 911 Call – audio,2/06,Michigan,1,1,1,0,"– In Feb. 2006 5 year-old Robert Turner called to say his mother was unconscious. However, dispatcher Sharon Nicols believed it was a prank call. Nicols and another dispatcher were later fired, and Nicols was charged with criminal neglect of duty. She was later convicted, but granted probation. Also isten to the second",call_2.mp3
8,https://web.archive.org/web/20150417085342/http://mp3.911dispatch.com.s3.amazonaws.com/hernlen_choufani_911.mp3,Girl’s Murder 911 Call,3/05,Florida,1,2,1,0,"– the 911 call of a lifetime. Volusia County (Fla.) dispatcher Donna Choufani talks to 5 year-old Tia Hernlen, who calmly reports her two parents shot to death. Also check this",call_8.mp3
9,https://web.archive.org/web/20150417085342/http://mp3.911dispatch.com.s3.amazonaws.com/watauga_911.mp3,‘Shoot Her?’ 911 call,4/05,Texas,1,0,0,1,"– caller reports her daughter is creating a disturbance at home, and the dispatcher makes an inappropriate comment",call_9.mp3
10,https://web.archive.org/web/20150417085342/http://mp3.911dispatch.com.s3.amazonaws.com/wamsley_to_douglascounty.mp3,Snowstorm 911 Call,1/05,Nebraska,1,2,1,0,– a couple under the influence of drugs dialed 911 after their truck ran into a snowdrift outside Omaha (Neb.) in Jan. 2005. Janelle Hornickel and Michael Wamsley were disoriented and couldn’t give their location. Their bodies were found days later in the snow.,call_10.mp3


In [0]:
from pyspark.sql.functions import col, when, udf
from pyspark.sql.types import StringType

# Fill missing values
metadata_df = metadata_df \
    .fillna({"date": "unknown", "state": "unknown"}) \
    .fillna({"civilian_initiated": 0, "deaths": 0,
             "potential_death": 0, "false_alarm": 0})

# Keyword lists
fire_keywords     = ["fire", "arson", "smoke", "flame", "wildfire", 
                     "explosion", "refinery", "hazmat", "burning", "blaze"]

medical_keywords  = ["cpr", "choking", "unconscious", "overdose", "heart attack",
                     "baby delivery", "drowning", "not breathing", "medical",
                     "electrocution", "carbon monoxide", "heatstroke", "diabetes",
                     "crash", "plane crash", "helicopter crash", "train crash",
                     "collapse", "flood", "tornado", "rescue", "injured",
                     "trapped", "missing", "sinking", "delivery", "child birth"]

violence_keywords = ["murder", "shooting", "shot", "stabbing", "stab", "robbery",
                     "robber", "kidnap", "assault", "hostage", "homicide",
                     "weapon", "gun", "burglary", "burglar", "invasion",
                     "rape", "beating", "beheading", "carjack", "abduction",
                     "pursuit", "shoot-out", "armed", "intruder", "gunman", "bomb"]

irrelevant = ["wheel of fortune", "wacky 911", "adam-12", "fcc 1st resp",
              "leno's wacky", "non-emerg", "onstar", "bad sandwich",
              "fried rice", "mcnugget", "burger king", "bad chicken",
              "kissing in restaurant", "joe mccain", "who wants to be",
              "tape archive", "sept. 11th tape archive"]

def classify(title, description):
    if title is None:
        return "unknown"
    title_lower = title.lower()
    for t in irrelevant:
        if t in title_lower:
            return "unknown"
    text = (title + " " + (description or "")).lower()
    fire_score     = sum(1 for k in fire_keywords     if k in text)
    medical_score  = sum(1 for k in medical_keywords  if k in text)
    violence_score = sum(1 for k in violence_keywords if k in text)
    if fire_score == 0 and medical_score == 0 and violence_score == 0:
        return "unknown"
    scores = {"fire": fire_score, "medical": medical_score, "violence": violence_score}
    return max(scores, key=scores.get)

classify_udf = udf(classify, StringType())

# Derive labels
metadata_df = metadata_df.withColumn("label", classify_udf(col("title"), col("description")))

# Drop unknowns
metadata_df = metadata_df.filter(col("label") != "unknown")

# Encode labels
metadata_df = metadata_df.withColumn(
    "label_encoded",
    when(col("label") == "medical",   0)
    .when(col("label") == "fire",     1)
    .when(col("label") == "violence", 2)
)

print("=== LABEL DISTRIBUTION ===")
metadata_df.groupBy("label", "label_encoded").count().show()
print(f"Total labeled rows: {metadata_df.count()}")

=== LABEL DISTRIBUTION ===
+--------+-------------+-----+
|   label|label_encoded|count|
+--------+-------------+-----+
|    fire|            1|   50|
|violence|            2|  432|
| medical|            0|  140|
+--------+-------------+-----+

Total labeled rows: 622


In [0]:
import librosa
import numpy as np
import io

def extract_features(file_name):
    try:
        # Read audio bytes directly from ADLS into memory
        audio_path = f"abfss://raw@{storage_account_name}.dfs.core.windows.net/911-recordings/v1/audio/{file_name}"
        audio_bytes = spark.read.format("binaryFile").load(audio_path).first().content
        
        # Load into librosa from memory
        y, sr = librosa.load(io.BytesIO(bytes(audio_bytes)), sr=16000, mono=True)
        
        # VAD - find first speech onset using RMS energy
        hop_length    = 512
        frame_length  = 2048
        rms           = librosa.feature.rms(y=y, frame_length=frame_length, hop_length=hop_length)[0]
        threshold     = np.mean(rms) * 0.5
        speech_frames = np.where(rms > threshold)[0]
        
        if len(speech_frames) == 0:
            return None
        
        # Trim to 30s from first speech onset
        start_sample = speech_frames[0] * hop_length
        end_sample   = min(start_sample + (30 * sr), len(y))
        y_trimmed    = y[start_sample:end_sample]
        
        if len(y_trimmed) < sr * 5:
            return None
        
        # Extract features
        mfccs              = librosa.feature.mfcc(y=y_trimmed, sr=sr, n_mfcc=13)
        delta_mfccs        = librosa.feature.delta(mfccs)
        zcr                = librosa.feature.zero_crossing_rate(y_trimmed)
        spectral_centroid  = librosa.feature.spectral_centroid(y=y_trimmed, sr=sr)
        spectral_rolloff   = librosa.feature.spectral_rolloff(y=y_trimmed, sr=sr)
        spectral_bandwidth = librosa.feature.spectral_bandwidth(y=y_trimmed, sr=sr)
        rms_energy         = librosa.feature.rms(y=y_trimmed)
        chroma             = librosa.feature.chroma_stft(y=y_trimmed, sr=sr)
        
        # Aggregate mean + std
        features = {}
        for i in range(13):
            features[f"mfcc_{i+1}_mean"]       = float(np.mean(mfccs[i]))
            features[f"mfcc_{i+1}_std"]        = float(np.std(mfccs[i]))
            features[f"delta_mfcc_{i+1}_mean"] = float(np.mean(delta_mfccs[i]))
            features[f"delta_mfcc_{i+1}_std"]  = float(np.std(delta_mfccs[i]))

        features["zcr_mean"]                = float(np.mean(zcr))
        features["zcr_std"]                 = float(np.std(zcr))
        features["spectral_centroid_mean"]  = float(np.mean(spectral_centroid))
        features["spectral_centroid_std"]   = float(np.std(spectral_centroid))
        features["spectral_rolloff_mean"]   = float(np.mean(spectral_rolloff))
        features["spectral_rolloff_std"]    = float(np.std(spectral_rolloff))
        features["spectral_bandwidth_mean"] = float(np.mean(spectral_bandwidth))
        features["spectral_bandwidth_std"]  = float(np.std(spectral_bandwidth))
        features["rms_energy_mean"]         = float(np.mean(rms_energy))
        features["rms_energy_std"]          = float(np.std(rms_energy))
        
        for i in range(12):
            features[f"chroma_{i+1}_mean"] = float(np.mean(chroma[i]))
            features[f"chroma_{i+1}_std"]  = float(np.std(chroma[i]))
        
        features["file_name"] = file_name
        return features

    except Exception as e:
        print(f"Failed: {file_name} — {e}")
        return None

print("extract_features function defined")

extract_features function defined


In [0]:
# Test on a single file
test_file = "call_1.mp3"
print(f"Testing with: {test_file}")

result = extract_features(test_file)

if result:
    print(f"Success! ✅")
    print(f"Number of features extracted: {len(result) - 1}")  # -1 for file_name
    print(f"Sample features:")
    for k, v in list(result.items())[:5]:
        print(f"  {k}: {v}")
else:
    print("Failed ❌")

Testing with: call_1.mp3
Success! ✅
Number of features extracted: 86
Sample features:
  mfcc_1_mean: -169.06163024902344
  mfcc_1_std: 44.980918884277344
  delta_mfcc_1_mean: 0.010393252596259117
  delta_mfcc_1_std: 7.082015037536621
  mfcc_2_mean: 96.1220703125


In [0]:
# Get list of labeled file names
labeled_files = [
    row["file_name"]
    for row in metadata_df.select("file_name").collect()
    if row["file_name"] is not None
]

print(f"Total files to process: {len(labeled_files)}")

BATCH_SIZE = 50
all_features = []
failed = []

for batch_start in range(0, len(labeled_files), BATCH_SIZE):
    batch = labeled_files[batch_start : batch_start + BATCH_SIZE]
    
    for file_name in batch:
        result = extract_features(file_name)
        if result:
            all_features.append(result)
        else:
            failed.append(file_name)
    
    batch_num = (batch_start // BATCH_SIZE) + 1
    total_batches = (len(labeled_files) + BATCH_SIZE - 1) // BATCH_SIZE
    print(f"Batch {batch_num}/{total_batches} done — extracted so far: {len(all_features)}")

print(f"\nSuccessfully extracted : {len(all_features)}")
print(f"Failed/skipped         : {len(failed)}")
if failed:
    print(f"Failed files: {failed[:10]}")

Total files to process: 622
Failed: call_69.mp3 — Error opening <_io.BytesIO object at 0x7f12721d7880>: Format not recognised.
Batch 1/13 done — extracted so far: 49


Failed: call_132.mp3 — Error opening <_io.BytesIO object at 0x7f1271be4270>: Format not recognised.


Batch 2/13 done — extracted so far: 98
Failed: call_184.mp3 — Error opening <_io.BytesIO object at 0x7f129cadba10>: Format not recognised.
Batch 3/13 done — extracted so far: 147
Failed: call_200.mp3 — Error opening <_io.BytesIO object at 0x7f129cadba10>: Format not recognised.


[src/libmpg123/id3.c:process_extra():684] error: No extra frame text / valid description?


Failed: call_239.mp3 — Error opening <_io.BytesIO object at 0x7f12d145b0b0>: Format not recognised.


...

Failed: call_477.mp3 — Error opening <_io.BytesIO object at 0x7f1271ea27a0>: Format not recognised.
Batch 8/13 done — extracted so far: 385
Failed: call_491.mp3 — Error opening <_io.BytesIO object at 0x7f1271f34c70>: Format not recognised.


Failed: call_506.mp3 — Error opening <_io.BytesIO object at 0x7f129c21a2a0>: Format not recognised.


Batch 9/13 done — extracted so far: 433


Failed: call_565.mp3 — Error opening <_io.BytesIO object at 0x7f128b0c98a0>: Format not recognised.
Failed: call_566.mp3 — Error opening <_io.BytesIO object at 0x7f12e9992ed0>: Format not recognised.
Failed: call_586.mp3 — Error opening <_io.BytesIO object at 0x7f1271be4270>: Format not recognised.
Batch 10/13 done — extracted so far: 480
Failed: call_602.mp3 — Error opening <_io.BytesIO object at 0x7f129c23f4c0>: Format not recognised.
Failed: call_606.mp3 — Error opening <_io.BytesIO object at 0x7f12e9992ed0>: Format not recognised.
Batch 11/13 done — extracted so far: 528
Failed: call_668.mp3 — Error opening <_io.BytesIO object at 0x7f129c23f4c0>: Format not recognised.
Failed: call_671.mp3 — Error opening <_io.BytesIO object at 0x7f1271be4270>: Format not recognised.


Failed: call_693.mp3 — Error opening <_io.BytesIO object at 0x7f1271be4270>: Format not recognised.
Failed: call_700.mp3 — Error opening <_io.BytesIO object at 0x7f1271be4270>: Format not recognised.
Failed: call_703.mp3 — Error opening <_io.BytesIO object at 0x7f1271be4270>: Format not recognised.
Failed: call_709.mp3 — Error opening <_io.BytesIO object at 0x7f1271be4270>: Format not recognised.
Failed: call_714.mp3 — Error opening <_io.BytesIO object at 0x7f1271be4270>: Format not recognised.
Batch 12/13 done — extracted so far: 571
Failed: call_727.mp3 — Error opening <_io.BytesIO object at 0x7f129c23f4c0>: Format not recognised.
Failed: call_737.mp3 — Error opening <_io.BytesIO object at 0x7f129c23f4c0>: Format not recognised.
Batch 13/13 done — extracted so far: 591

Successfully extracted : 591
Failed/skipped         : 31
Failed files: ['call_69.mp3', 'call_132.mp3', 'call_184.mp3', 'call_200.mp3', 'call_239.mp3', 'call_299.mp3', 'call_312.mp3', 'call_315.mp3', 'call_361.mp3', 'c

In [0]:
# Convert features list to Spark DataFrame
features_df = spark.createDataFrame(all_features)

print(f"Features DataFrame rows    : {features_df.count()}")
print(f"Features DataFrame columns : {len(features_df.columns)}")

# Merge with metadata labels
final_df = features_df.join(
    metadata_df.select("file_name", "label", "label_encoded",
                       "civilian_initiated", "deaths",
                       "potential_death", "false_alarm"),
    on="file_name",
    how="inner"
)

print(f"\nAfter merge with labels:")
print(f"Rows    : {final_df.count()}")
print(f"Columns : {len(final_df.columns)}")
display(final_df.limit(5))

Features DataFrame rows    : 591
Features DataFrame columns : 87

After merge with labels:
Rows    : 591
Columns : 93


file_name,chroma_10_mean,chroma_10_std,chroma_11_mean,chroma_11_std,chroma_12_mean,chroma_12_std,chroma_1_mean,chroma_1_std,chroma_2_mean,chroma_2_std,chroma_3_mean,chroma_3_std,chroma_4_mean,chroma_4_std,chroma_5_mean,chroma_5_std,chroma_6_mean,chroma_6_std,chroma_7_mean,chroma_7_std,chroma_8_mean,chroma_8_std,chroma_9_mean,chroma_9_std,delta_mfcc_10_mean,delta_mfcc_10_std,delta_mfcc_11_mean,delta_mfcc_11_std,delta_mfcc_12_mean,delta_mfcc_12_std,delta_mfcc_13_mean,delta_mfcc_13_std,delta_mfcc_1_mean,delta_mfcc_1_std,delta_mfcc_2_mean,delta_mfcc_2_std,delta_mfcc_3_mean,delta_mfcc_3_std,delta_mfcc_4_mean,delta_mfcc_4_std,delta_mfcc_5_mean,delta_mfcc_5_std,delta_mfcc_6_mean,delta_mfcc_6_std,delta_mfcc_7_mean,delta_mfcc_7_std,delta_mfcc_8_mean,delta_mfcc_8_std,delta_mfcc_9_mean,delta_mfcc_9_std,mfcc_10_mean,mfcc_10_std,mfcc_11_mean,mfcc_11_std,mfcc_12_mean,mfcc_12_std,mfcc_13_mean,mfcc_13_std,mfcc_1_mean,mfcc_1_std,mfcc_2_mean,mfcc_2_std,mfcc_3_mean,mfcc_3_std,mfcc_4_mean,mfcc_4_std,mfcc_5_mean,mfcc_5_std,mfcc_6_mean,mfcc_6_std,mfcc_7_mean,mfcc_7_std,mfcc_8_mean,mfcc_8_std,mfcc_9_mean,mfcc_9_std,rms_energy_mean,rms_energy_std,spectral_bandwidth_mean,spectral_bandwidth_std,spectral_centroid_mean,spectral_centroid_std,spectral_rolloff_mean,spectral_rolloff_std,zcr_mean,zcr_std,label,label_encoded,civilian_initiated,deaths,potential_death,false_alarm
call_1.mp3,0.40851375460624695,0.27664920687675476,0.3938099443912506,0.28279608488082886,0.39633724093437195,0.272915780544281,0.43286287784576416,0.28483760356903076,0.6474030017852783,0.3733883798122406,0.49620988965034485,0.2693248391151428,0.4211757779121399,0.2612241804599762,0.41127604246139526,0.2898719310760498,0.43236443400382996,0.30209270119667053,0.45909154415130615,0.2639809846878052,0.5440878868103027,0.2737402319908142,0.5533125400543213,0.30495309829711914,0.0073159122839570045,1.1424115896224976,0.0072639090940356255,0.9597647190093994,0.010646634735167027,1.0245760679244995,-0.0015483888564631343,0.9529129862785339,0.010393252596259117,7.082015037536621,-0.030700528994202614,2.430131435394287,-0.007896613329648972,2.040696859359741,0.057745564728975296,2.450098991394043,-0.001298743300139904,1.70284903049469,-0.0435418002307415,1.9564018249511719,-0.019239384680986404,1.5871883630752563,-0.007623478304594755,1.4355943202972412,0.011340446770191193,1.1623034477233887,-0.5841963291168213,6.136508941650391,-8.792448043823242,5.111673831939697,4.2279815673828125,6.049551010131836,2.456557035446167,5.471254825592041,-169.06163024902344,44.980918884277344,96.1220703125,15.295633316040039,-43.591739654541016,13.26626205444336,-11.475350379943848,13.305560111999512,-17.791391372680664,11.321602821350098,-23.052959442138672,12.181950569152832,-16.11138343811035,11.032234191894531,-11.013318061828613,7.74224853515625,-13.469555854797363,7.5173258781433105,0.07633429020643234,0.06910298764705658,1524.7930725205774,221.01855427885013,1605.9779315494525,294.4941363803127,2818.1636460554373,675.9097114793125,0.13423778151652452,0.03368566396415983,violence,2,0,2,1,0
call_2.mp3,0.4315251410007477,0.32994699478149414,0.46286237239837646,0.35271981358528137,0.4503573775291443,0.34558767080307007,0.4124525785446167,0.34127718210220337,0.3814181983470917,0.33051037788391113,0.369579017162323,0.31644585728645325,0.365384578704834,0.31826987862586975,0.35623085498809814,0.33638477325439453,0.31908851861953735,0.29048582911491394,0.3618312180042267,0.3074386417865753,0.4260029196739197,0.33492687344551086,0.45705482363700867,0.3404794931411743,-0.014738569967448711,1.762258768081665,-0.002656651893630624,1.9910576343536377,0.024201774969697,1.3503309488296509,0.0012632672442123294,1.2924202680587769,0.12421902269124985,18.252729415893555,-0.0015614799922332168,9.191764831542969,-0.055527545511722565,5.9374589920043945,0.012896944768726826,3.893054723739624,-0.022805219516158104,2.4566738605499268,-0.05614805966615677,2.797368049621582,-0.010231428779661655,2.099186658859253,0.0097007611766

In [0]:
from pyspark.ml.feature import MinMaxScaler, VectorAssembler
from pyspark.ml import Pipeline
from pyspark.ml.functions import vector_to_array
from pyspark.sql.functions import col

# Only normalize acoustic feature columns
feature_cols = [c for c in features_df.columns if c != "file_name"]

print(f"Columns to normalize: {len(feature_cols)}")

# Assemble into vector
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features_vec")
scaler    = MinMaxScaler(inputCol="features_vec", outputCol="scaled_vec")
pipeline  = Pipeline(stages=[assembler, scaler])

# Fit and transform
scaler_model = pipeline.fit(final_df)
scaled_df    = scaler_model.transform(final_df)

# Unpack scaled vector back to individual columns
scaled_df  = scaled_df.withColumn("scaled_array", vector_to_array("scaled_vec"))
scaled_cols = [f"{c}_scaled" for c in feature_cols]

for i, c in enumerate(scaled_cols):
    scaled_df = scaled_df.withColumn(c, col("scaled_array")[i])

# Keep only final columns
cols_to_keep  = ["file_name", "label", "label_encoded",
                 "civilian_initiated", "deaths",
                 "potential_death", "false_alarm"] + scaled_cols

final_scaled_df = scaled_df.select(cols_to_keep)

print(f"Final rows    : {final_scaled_df.count()}")
print(f"Final columns : {len(final_scaled_df.columns)}")
display(final_scaled_df.limit(5))

Columns to normalize: 86
Final rows    : 591
Final columns : 93


file_name,label,label_encoded,civilian_initiated,deaths,potential_death,false_alarm,chroma_10_mean_scaled,chroma_10_std_scaled,chroma_11_mean_scaled,chroma_11_std_scaled,chroma_12_mean_scaled,chroma_12_std_scaled,chroma_1_mean_scaled,chroma_1_std_scaled,chroma_2_mean_scaled,chroma_2_std_scaled,chroma_3_mean_scaled,chroma_3_std_scaled,chroma_4_mean_scaled,chroma_4_std_scaled,chroma_5_mean_scaled,chroma_5_std_scaled,chroma_6_mean_scaled,chroma_6_std_scaled,chroma_7_mean_scaled,chroma_7_std_scaled,chroma_8_mean_scaled,chroma_8_std_scaled,chroma_9_mean_scaled,chroma_9_std_scaled,delta_mfcc_10_mean_scaled,delta_mfcc_10_std_scaled,delta_mfcc_11_mean_scaled,delta_mfcc_11_std_scaled,delta_mfcc_12_mean_scaled,delta_mfcc_12_std_scaled,delta_mfcc_13_mean_scaled,delta_mfcc_13_std_scaled,delta_mfcc_1_mean_scaled,delta_mfcc_1_std_scaled,delta_mfcc_2_mean_scaled,delta_mfcc_2_std_scaled,delta_mfcc_3_mean_scaled,delta_mfcc_3_std_scaled,delta_mfcc_4_mean_scaled,delta_mfcc_4_std_scaled,delta_mfcc_5_mean_scaled,delta_mfcc_5_std_scaled,delta_mfcc_6_mean_scaled,delta_mfcc_6_std_scaled,delta_mfcc_7_mean_scaled,delta_mfcc_7_std_scaled,delta_mfcc_8_mean_scaled,delta_mfcc_8_std_scaled,delta_mfcc_9_mean_scaled,delta_mfcc_9_std_scaled,mfcc_10_mean_scaled,mfcc_10_std_scaled,mfcc_11_mean_scaled,mfcc_11_std_scaled,mfcc_12_mean_scaled,mfcc_12_std_scaled,mfcc_13_mean_scaled,mfcc_13_std_scaled,mfcc_1_mean_scaled,mfcc_1_std_scaled,mfcc_2_mean_scaled,mfcc_2_std_scaled,mfcc_3_mean_scaled,mfcc_3_std_scaled,mfcc_4_mean_scaled,mfcc_4_std_scaled,mfcc_5_mean_scaled,mfcc_5_std_scaled,mfcc_6_mean_scaled,mfcc_6_std_scaled,mfcc_7_mean_scaled,mfcc_7_std_scaled,mfcc_8_mean_scaled,mfcc_8_std_scaled,mfcc_9_mean_scaled,mfcc_9_std_scaled,rms_energy_mean_scaled,rms_energy_std_scaled,spectral_bandwidth_mean_scaled,spectral_bandwidth_std_scaled,spectral_centroid_mean_scaled,spectral_centroid_std_scaled,spectral_rolloff_mean_scaled,spectral_rolloff_std_scaled,zcr_mean_scaled,zcr_std_scaled
call_1.mp3,violence,2,0,2,1,0,0.3655909561546461,0.3604957717549262,0.36814000074401326,0.38100163518752633,0.355429852156085,0.287046348260488,0.4009371228443312,0.47171002535467005,0.7005825992172732,0.774923349825164,0.48911478320811747,0.43993696798618936,0.4206483514889418,0.3969179818005344,0.4694755289264809,0.572094623963163,0.564312859349649,0.6027946495043145,0.5663865031173565,0.3430195258420491,0.6278459342888815,0.35770555729324816,0.6065088258573027,0.4701462116104671,0.5774339905844081,0.07086590741509464,0.5015927321595348,0.07488640499286016,0.5502178179869373,0.050366654981397856,0.5674155182297528,0.06574067322805663,0.6894850698561507,0.1567550163659877,0.5923342658469056,0.0188867768445445,0.41100022284298854,0.008068978560841445,0.6998873776534893,0.08289162329981523,0.49495438515215484,0.15552147553886353,0.39544602185452005,0.1611382328039721,0.396867145796345,0.0789529769358077,0.5090535364557321,0.07454088777443332,0.5341123061025647,0.02282239181196241,0.509103062560863,0.06577766535922303,0.4541825356947992,0.026901258736040247,0.6148395385553515,0.051100398974666145,0.5950401731923657,0.02932969208495445,0.7121641920287118,0.13502272911987095,0.3711283774646082,0.016016857547077188,0.6179716101854276,0.035455952666417434,0.5864256174372439,0.06591527683589422,0.42807237228163064,0.19724809736428348,0.5632031206365721,0.15364058293120642,0.3783538585602173,0.1378350379908391,0.5977535115114941,0.013504341768720342,0.497965365755969,0.04754188517625613,0.17991755388845218,0.2765665156574374,0.6524458115792455,0.23089519509432999,0.5114144901428583,0.16819232803980497,0.45770783353205674,0.24306647592808867,0.43289069929504537,0.09716946070240137
call_2.mp3,fire,1,1,1,1,0,0.4001603571706713,0.6262170287408039,0.46487904753948106,0.744243480382549,0.42978406220482607,0.640280140188879,0.37276947587454046,0.7265347527714472,0.33763233319413755,0.6150448255311896,0.3120608408317049,0.6358272004028023,0.335651813135943,0.6320026203811269,0.37748824989645885,0.781662427763982

In [0]:
from pyspark.sql.functions import isnan, when, count as spark_count

print("=== FINAL VALIDATION ===")
print(f"Total rows    : {final_scaled_df.count()}")
print(f"Total columns : {len(final_scaled_df.columns)}")

print("\n=== LABEL DISTRIBUTION ===")
final_scaled_df.groupBy("label", "label_encoded").count().show()

print("\n=== NULL CHECK ===")
final_scaled_df.select([
    spark_count(when(col(c).isNull(), c)).alias(c)
    for c in ["label", "label_encoded"] + scaled_cols[:5]
]).show()

print("\n=== SCALED FEATURE STATS (should be between 0 and 1) ===")
final_scaled_df.describe(scaled_cols[:5]).show()

=== FINAL VALIDATION ===
Total rows    : 591
Total columns : 93

=== LABEL DISTRIBUTION ===
+--------+-------------+-----+
|   label|label_encoded|count|
+--------+-------------+-----+
|    fire|            1|   48|
|violence|            2|  410|
| medical|            0|  133|
+--------+-------------+-----+


=== NULL CHECK ===
+-----+-------------+---------------------+--------------------+---------------------+--------------------+---------------------+
|label|label_encoded|chroma_10_mean_scaled|chroma_10_std_scaled|chroma_11_mean_scaled|chroma_11_std_scaled|chroma_12_mean_scaled|
+-----+-------------+---------------------+--------------------+---------------------+--------------------+---------------------+
|    0|            0|                    0|                   0|                    0|                   0|                    0|
+-----+-------------+---------------------+--------------------+---------------------+--------------------+---------------------+


=== SCALED FEATURE

In [0]:
silver_output_path = f"abfss://processed@{storage_account_name}.dfs.core.windows.net/911-recordings/features_silver"

final_scaled_df.write.mode("overwrite").parquet(silver_output_path)

print(f"Silver layer written successfully")
print(f"Path: {silver_output_path}")

# Read back and verify
silver_check = spark.read.parquet(silver_output_path)

print("\n=== SILVER LAYER VERIFICATION ===")
print(f"Rows    : {silver_check.count()}")
print(f"Columns : {len(silver_check.columns)}")
silver_check.printSchema()
display(silver_check.limit(5))

Silver layer written successfully
Path: abfss://processed@azuredatalake60304739.dfs.core.windows.net/911-recordings/features_silver

=== SILVER LAYER VERIFICATION ===
Rows    : 591
Columns : 93
root
 |-- file_name: string (nullable = true)
 |-- label: string (nullable = true)
 |-- label_encoded: integer (nullable = true)
 |-- civilian_initiated: integer (nullable = true)
 |-- deaths: integer (nullable = true)
 |-- potential_death: integer (nullable = true)
 |-- false_alarm: integer (nullable = true)
 |-- chroma_10_mean_scaled: double (nullable = true)
 |-- chroma_10_std_scaled: double (nullable = true)
 |-- chroma_11_mean_scaled: double (nullable = true)
 |-- chroma_11_std_scaled: double (nullable = true)
 |-- chroma_12_mean_scaled: double (nullable = true)
 |-- chroma_12_std_scaled: double (nullable = true)
 |-- chroma_1_mean_scaled: double (nullable = true)
 |-- chroma_1_std_scaled: double (nullable = true)
 |-- chroma_2_mean_scaled: double (nullable = true)
 |-- chroma_2_std_scaled:

file_name,label,label_encoded,civilian_initiated,deaths,potential_death,false_alarm,chroma_10_mean_scaled,chroma_10_std_scaled,chroma_11_mean_scaled,chroma_11_std_scaled,chroma_12_mean_scaled,chroma_12_std_scaled,chroma_1_mean_scaled,chroma_1_std_scaled,chroma_2_mean_scaled,chroma_2_std_scaled,chroma_3_mean_scaled,chroma_3_std_scaled,chroma_4_mean_scaled,chroma_4_std_scaled,chroma_5_mean_scaled,chroma_5_std_scaled,chroma_6_mean_scaled,chroma_6_std_scaled,chroma_7_mean_scaled,chroma_7_std_scaled,chroma_8_mean_scaled,chroma_8_std_scaled,chroma_9_mean_scaled,chroma_9_std_scaled,delta_mfcc_10_mean_scaled,delta_mfcc_10_std_scaled,delta_mfcc_11_mean_scaled,delta_mfcc_11_std_scaled,delta_mfcc_12_mean_scaled,delta_mfcc_12_std_scaled,delta_mfcc_13_mean_scaled,delta_mfcc_13_std_scaled,delta_mfcc_1_mean_scaled,delta_mfcc_1_std_scaled,delta_mfcc_2_mean_scaled,delta_mfcc_2_std_scaled,delta_mfcc_3_mean_scaled,delta_mfcc_3_std_scaled,delta_mfcc_4_mean_scaled,delta_mfcc_4_std_scaled,delta_mfcc_5_mean_scaled,delta_mfcc_5_std_scaled,delta_mfcc_6_mean_scaled,delta_mfcc_6_std_scaled,delta_mfcc_7_mean_scaled,delta_mfcc_7_std_scaled,delta_mfcc_8_mean_scaled,delta_mfcc_8_std_scaled,delta_mfcc_9_mean_scaled,delta_mfcc_9_std_scaled,mfcc_10_mean_scaled,mfcc_10_std_scaled,mfcc_11_mean_scaled,mfcc_11_std_scaled,mfcc_12_mean_scaled,mfcc_12_std_scaled,mfcc_13_mean_scaled,mfcc_13_std_scaled,mfcc_1_mean_scaled,mfcc_1_std_scaled,mfcc_2_mean_scaled,mfcc_2_std_scaled,mfcc_3_mean_scaled,mfcc_3_std_scaled,mfcc_4_mean_scaled,mfcc_4_std_scaled,mfcc_5_mean_scaled,mfcc_5_std_scaled,mfcc_6_mean_scaled,mfcc_6_std_scaled,mfcc_7_mean_scaled,mfcc_7_std_scaled,mfcc_8_mean_scaled,mfcc_8_std_scaled,mfcc_9_mean_scaled,mfcc_9_std_scaled,rms_energy_mean_scaled,rms_energy_std_scaled,spectral_bandwidth_mean_scaled,spectral_bandwidth_std_scaled,spectral_centroid_mean_scaled,spectral_centroid_std_scaled,spectral_rolloff_mean_scaled,spectral_rolloff_std_scaled,zcr_mean_scaled,zcr_std_scaled
call_556.mp3,violence,2,1,1,1,0,0.3502798616989358,0.6477256611153082,0.31935490295324404,0.573957788567559,0.2825566429815504,0.4663061991888468,0.26650078672122496,0.6069430056136044,0.23067371062385145,0.4938832654869135,0.22242623738574202,0.5331631698576242,0.31800611947725305,0.6110551688724857,0.3615878840809181,0.6819942129276841,0.35739507013650285,0.5891908121686559,0.3113786632631488,0.47647181712419473,0.3112303990115843,0.5190562935481442,0.3832052817169792,0.6203354701211383,0.4381275415990389,0.20218491784231374,0.6245111522179798,0.1703960417948647,0.5707436001497371,0.14127785603041162,0.5019256543667581,0.18700804326606166,0.6879649929934862,0.22785230142431975,0.6910022286515181,0.2132438979560844,0.41176857285428003,0.294989341071356,0.7310748563285677,0.24553680047570461,0.4836442820072955,0.3131348243855731,0.25074822062291874,0.33819643529921456,0.46072619107768337,0.1661334468199109,0.5867832787616241,0.21570350318671927,0.4564915310644629,0.25141482724825565,0.5317649582970603,0.19294042085230195,0.38215129903581274,0.16475081164666858,0.7173189386818286,0.14711423124753178,0.30085354573856726,0.12109208238231833,0.8555442639216844,0.17156279593843926,0.8028261019862682,0.1273250275448768,0.32369817174843074,0.2382175216340236,0.938308212379535,0.220884006791406,0.39297534272569673,0.37141791957714343,0.586127072468818,0.2681124988253241,0.8542536340887439,0.16574896374420292,0.39667907466283614,0.2195728011535138,0.8068979776935477,0.2121888147481699,0.5471761074690183,0.31634011150171626,0.3436167599187624,0.12349221186854378,0.3741671714573376,0.21077942023906943,0.3788194394564077,0.18174208838927605,0.3938756338239654,0.22034622782672636
call_557.mp3,violence,2,1,1,1,0,0.2359794534903876,0.6224387281677738,0.20062496276280586,0.5189986434848413,0.13842571092010686,0.3216885418450245,0.11867908486323356,0.3551845745433085,0.1886109836940011,0.491368430784706,0.19826092380776397,0.6236326193521019,0.16816607805689923,0.4656969684464895,0.2604048653486842,0.685132670425